### DDL Export Schema from Gaia SQLite

```sql
-- accounts definition

CREATE TABLE "accounts" (
    "id" TEXT PRIMARY KEY,
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "name" TEXT,
    "username" TEXT,
    "email" TEXT NOT NULL,
    "avatarUrl" TEXT,
    "details" TEXT DEFAULT '{}' CHECK(json_valid("details")) -- Ensuring details is a valid JSON field
);


-- cache definition

CREATE TABLE "cache" (
    "key" TEXT NOT NULL,
    "agentId" TEXT NOT NULL,
    "value" TEXT DEFAULT '{}' CHECK(json_valid("value")),
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "expiresAt" TIMESTAMP,
    PRIMARY KEY ("key", "agentId")
);


-- goals definition

CREATE TABLE "goals" (
    "id" TEXT PRIMARY KEY,
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "userId" TEXT,
    "name" TEXT,
    "status" TEXT,
    "description" TEXT,
    "roomId" TEXT,
    "objectives" TEXT DEFAULT '[]' NOT NULL CHECK(json_valid("objectives")) -- Ensuring objectives is a valid JSON array
);


-- logs definition

CREATE TABLE "logs" (
    "id" TEXT PRIMARY KEY,
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "userId" TEXT NOT NULL,
    "body" TEXT NOT NULL,
    "type" TEXT NOT NULL,
    "roomId" TEXT NOT NULL
);


-- rooms definition

CREATE TABLE "rooms" (
    "id" TEXT PRIMARY KEY,
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


-- sqlite_schema definition

CREATE TABLE sqlite_schema (
	"type" TEXT,
	name TEXT,
	tbl_name TEXT,
	rootpage INT,
	"sql" TEXT
);


-- knowledge definition

CREATE TABLE "knowledge" (
    "id" TEXT PRIMARY KEY,
    "agentId" TEXT,
    "content" TEXT NOT NULL CHECK(json_valid("content")),
    "embedding" BLOB,
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "isMain" INTEGER DEFAULT 0,
    "originalId" TEXT,
    "chunkIndex" INTEGER,
    "isShared" INTEGER DEFAULT 0,
    FOREIGN KEY ("agentId") REFERENCES "accounts"("id"),
    FOREIGN KEY ("originalId") REFERENCES "knowledge"("id"),
    CHECK((isShared = 1 AND agentId IS NULL) OR (isShared = 0 AND agentId IS NOT NULL))
);

CREATE INDEX "knowledge_agent_key" ON "knowledge" ("agentId");
CREATE INDEX "knowledge_agent_main_key" ON "knowledge" ("agentId", "isMain");
CREATE INDEX "knowledge_original_key" ON "knowledge" ("originalId");
CREATE INDEX "knowledge_content_key" ON "knowledge"
    ((json_extract(content, '$.text')))
    WHERE json_extract(content, '$.text') IS NOT NULL;
CREATE INDEX "knowledge_created_key" ON "knowledge" ("agentId", "createdAt");
CREATE INDEX "knowledge_shared_key" ON "knowledge" ("isShared");


-- memories definition

CREATE TABLE "memories" (
    "id" TEXT PRIMARY KEY,
    "type" TEXT NOT NULL,
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "content" TEXT NOT NULL,
    "embedding" BLOB NOT NULL, -- TODO: EMBEDDING ARRAY, CONVERT TO BEST FORMAT FOR SQLITE-VSS (JSON?)
    "userId" TEXT,
    "roomId" TEXT,
    "agentId" TEXT,
    "unique" INTEGER DEFAULT 1 NOT NULL,
    FOREIGN KEY ("userId") REFERENCES "accounts"("id"),
    FOREIGN KEY ("roomId") REFERENCES "rooms"("id"),
    FOREIGN KEY ("agentId") REFERENCES "accounts"("id")
);

CREATE UNIQUE INDEX "memories_id_key" ON "memories" ("id");


-- participants definition

CREATE TABLE "participants" (
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "userId" TEXT,
    "roomId" TEXT,
    "userState" TEXT,
    "id" TEXT PRIMARY KEY,
    "last_message_read" TEXT,
    FOREIGN KEY ("userId") REFERENCES "accounts"("id"),
    FOREIGN KEY ("roomId") REFERENCES "rooms"("id")
);

CREATE UNIQUE INDEX "participants_id_key" ON "participants" ("id");


-- relationships definition

CREATE TABLE "relationships" (
    "createdAt" TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    "userA" TEXT NOT NULL,
    "userB" TEXT NOT NULL,
    "status" "text",
    "id" TEXT PRIMARY KEY,
    "userId" TEXT NOT NULL,
    FOREIGN KEY ("userA") REFERENCES "accounts"("id"),
    FOREIGN KEY ("userB") REFERENCES "accounts"("id"),
    FOREIGN KEY ("userId") REFERENCES "accounts"("id")
);

CREATE UNIQUE INDEX "relationships_id_key" ON "relationships" ("id");
```

### Analyzing Gaia Database in Python

Import dependencies:

In [1]:
import sqlite3
import os
import sys
from pathlib import Path
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, MetaData, Table
from datetime import datetime

# Display all columns
pd.set_option('display.max_columns', None)
# Display 100 rows
pd.set_option('display.max_rows', 100)

Adding the Agent Database to our path:

In [2]:
notebook_dir = Path.cwd()  # Gets the current notebook directory
project_root = notebook_dir.parent  # Goes up one level to project root
sys.path.append(str(project_root))

# Define the database path
db_path = project_root / "agent" / "data" / "db.sqlite"

Connect to Gaia DB using sqlite3:

In [3]:
# Create a direct sqlite3 connection first to test
conn = sqlite3.connect(str(db_path))

In [4]:
def create_relationship_matrix(conn):
   # Get all tables
   tables_query = """
   SELECT name 
   FROM sqlite_master 
   WHERE type='table' 
   AND name NOT LIKE 'sqlite_%'
   ORDER BY name
   """
   tables = pd.read_sql_query(tables_query, conn)['name'].tolist()
   
   # Initialize matrix with empty strings
   matrix = pd.DataFrame(
       '',
       index=tables,
       columns=tables
   )
   
   # For each table, get its foreign keys
   relations = {}  # Track relations between table pairs
   
   for table in tables:
       fk_query = f"PRAGMA foreign_key_list('{table}')"
       foreign_keys = pd.read_sql_query(fk_query, conn)
       
       # Group by referenced table to collect all columns referencing it
       for to_table in foreign_keys['table'].unique():
           to_cols = foreign_keys[foreign_keys['table'] == to_table]['to'].unique()
           from_cols = foreign_keys[foreign_keys['table'] == to_table]['from'].tolist()
           
           # For each unique to_col, collect all from_cols pointing to it
           for to_col in to_cols:
               matching_from_cols = foreign_keys[
                   (foreign_keys['table'] == to_table) & 
                   (foreign_keys['to'] == to_col)
               ]['from'].tolist()
               
               # Sort the from_cols for consistent output
               matching_from_cols.sort()
               
               # Create the relationship strings
               forward = f"→ {to_table}({to_col})"
               backward = f"← {table}({','.join(matching_from_cols)})"
               
               matrix.loc[table, to_table] = forward
               matrix.loc[to_table, table] = backward
   
   return matrix

relationship_matrix = create_relationship_matrix(conn)
relationship_matrix

,accounts,cache,goals,knowledge,logs,memories,participants,relationships,rooms
accounts,,,,← knowledge(agentId),,"← memories(agentId,userId)",← participants(userId),"← relationships(userA,userB,userId)",
cache,,,,,,,,,
goals,,,,,,,,,
knowledge,→ accounts(id),,,← knowledge(originalId),,,,,
logs,,,,,,,,,
memories,→ accounts(id),,,,,,,,→ rooms(id)
participants,→ accounts(id),,,,,,,,→ rooms(id)
relationships,→ accounts(id),,,,,,,,
rooms,,,,,,← memories(roomId),← participants(roomId),,


In [5]:
# Query to get table info, columns, and foreign keys
def get_db_schema(conn):
    # Get all tables
    tables_query = """
    SELECT name 
    FROM sqlite_master 
    WHERE type='table' 
    AND name NOT LIKE 'sqlite_%'
    """
    tables = pd.read_sql_query(tables_query, conn)
    
    schema_data = []
    
    for table_name in tables['name']:
        # Get column info
        columns = pd.read_sql_query(f"PRAGMA table_info('{table_name}')", conn)
        
        # Get foreign keys
        foreign_keys = pd.read_sql_query(f"PRAGMA foreign_key_list('{table_name}')", conn)
        
        # Get indexes
        indexes = pd.read_sql_query(f"PRAGMA index_list('{table_name}')", conn)
        
        # Process each column
        for _, col in columns.iterrows():
            row = {
                'table_name': table_name,
                'column_name': col['name'],
                'data_type': col['type'],
                'nullable': not bool(col['notnull']),
                'primary_key': bool(col['pk']),
                'default_value': col['dflt_value']
            }
            
            # Add foreign key info if exists
            fk = foreign_keys[foreign_keys['from'] == col['name']]
            if not fk.empty:
                row['references_table'] = fk.iloc[0]['table']
                row['references_column'] = fk.iloc[0]['to']
            else:
                row['references_table'] = None
                row['references_column'] = None
            
            schema_data.append(row)
    
    return pd.DataFrame(schema_data)

# Create the schema DataFrame
df_schema = get_db_schema(conn)

# Sort by table name and primary key (to show PKs first)
df_schema = df_schema.sort_values(['table_name', 'primary_key'], ascending=[True, False])

print("Database Schema Analysis:")
print(f"Total Tables: {len(df_schema['table_name'].unique())}")
print(f"Total Columns: {len(df_schema)}")
print(f"Total Foreign Keys: {df_schema['references_table'].notna().sum()}")

Database Schema Analysis:
Total Tables: 9
Total Columns: 58
Total Foreign Keys: 10


In [6]:
df_schema

,table_name,column_name,data_type,nullable,primary_key,default_value,references_table,references_column
0,accounts,id,TEXT,True,True,None,None,None
1,accounts,createdAt,TIMESTAMP,True,False,CURRENT_TIMESTAMP,None,None
2,accounts,name,TEXT,True,False,None,None,None
3,accounts,username,TEXT,True,False,None,None,None
4,accounts,email,TEXT,False,False,None,None,None
5,accounts,avatarUrl,TEXT,True,False,None,None,None
6,accounts,details,TEXT,True,False,'{}',None,None
44,cache,key,TEXT,False,True,None,None,None
45,cache,agentId,TEXT,False,True,None,None,None
46,cache,value,TEXT,True,False,'{}',None,None


In [7]:
# Optional: Get table counts
counts_query = """
SELECT 
    'accounts' as table_name, COUNT(*) as row_count FROM accounts UNION ALL
    SELECT 'cache', COUNT(*) FROM cache UNION ALL
    SELECT 'goals', COUNT(*) FROM goals UNION ALL
    SELECT 'knowledge', COUNT(*) FROM knowledge UNION ALL
    SELECT 'logs', COUNT(*) FROM logs UNION ALL
    SELECT 'memories', COUNT(*) FROM memories UNION ALL
    SELECT 'participants', COUNT(*) FROM participants UNION ALL
    SELECT 'relationships', COUNT(*) FROM relationships UNION ALL
    SELECT 'rooms', COUNT(*) FROM rooms
"""
df_counts = pd.read_sql_query(counts_query, conn)
df_counts

,table_name,row_count
0,accounts,485
1,cache,1530
2,goals,0
3,knowledge,35299
4,logs,250
5,memories,8386
6,participants,1589
7,relationships,0
8,rooms,710


In [8]:
# Optional: Get table counts
table_query = """
SELECT 
    m.tbl_name as table_name,
    m.type as object_type,
    (SELECT COUNT(*) FROM sqlite_master WHERE type='index' AND tbl_name=m.tbl_name) as index_count,
    (SELECT COUNT(*) FROM pragma_table_info(m.tbl_name)) as column_count,
    CASE 
        WHEN m.sql LIKE '%BLOB%' THEN 1 
        ELSE 0 
    END as has_blob,
    CASE 
        WHEN m.sql LIKE '%json_valid%' THEN 1
        ELSE 0
    END as has_json
FROM sqlite_master m
WHERE m.type = 'table'
AND m.tbl_name NOT LIKE 'sqlite_%'
ORDER BY table_name;
"""
df_tables = pd.read_sql_query(table_query, conn)
df_tables

,table_name,object_type,index_count,column_count,has_blob,has_json
0,accounts,table,1,7,0,1
1,cache,table,1,5,0,1
2,goals,table,1,8,0,1
3,knowledge,table,7,9,1,1
4,logs,table,1,6,0,0
5,memories,table,2,9,1,0
6,participants,table,2,6,0,0
7,relationships,table,2,6,0,0
8,rooms,table,1,2,0,0


In [9]:
# Get all table names first
tables_query = """
SELECT name FROM sqlite_master 
WHERE type='table' 
AND name NOT LIKE 'sqlite_%';
"""
tables = pd.read_sql_query(tables_query, conn)['name']

# Load each table into a dictionary of dataframes and get their info
dfs = {}
table_stats = []

for table in tables:
    try:
        # Load table
        df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
        dfs[table] = df
        
        # Calculate memory usage
        memory_bytes = df.memory_usage(deep=True).sum()
        
        table_stats.append({
            'table': table,
            'rows': len(df),
            'columns': len(df.columns),
            'memory_bytes': memory_bytes,
            'memory_mb': memory_bytes / (1024 * 1024)
        })
    except Exception as e:
        print(f"Error loading table {table}: {e}")

# Convert stats to dataframe
stats_df = pd.DataFrame(table_stats)
if not stats_df.empty:
    stats_df['pct_total_memory'] = (stats_df['memory_bytes'] / stats_df['memory_bytes'].sum() * 100).round(2)
    stats_df = stats_df.sort_values('memory_bytes', ascending=False)

In [10]:
stats_df

,table,rows,columns,memory_bytes,memory_mb,pct_total_memory
7,cache,1530,5,378082743,360.567801,51.18
8,knowledge,35299,9,310783610,296.386347,42.07
1,memories,8386,9,32460742,30.956976,4.39
3,logs,250,6,16352460,15.594921,2.21
4,participants,1589,6,640495,0.610824,0.09
0,accounts,485,7,245657,0.234277,0.03
6,rooms,710,2,120118,0.114553,0.02
2,goals,0,8,124,0.000118,0.00
5,relationships,0,6,124,0.000118,0.00


In [11]:
# def analyze_table(df, table_name):
#     print(f"\n{'='*20} {table_name} Analysis {'='*20}")
    
#     # Basic stats
#     print(f"\nBasic Information:")
#     print(f"Total rows: {len(df)}")
#     print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
#     print(f"Columns: {', '.join(df.columns)}")
    
#     # Column-by-column analysis
#     print("\nColumn Details:")
#     for col in df.columns:
#         print(f"\n--- {col} ---")
#         print(f"Type: {df[col].dtype}")
#         print(f"Null values: {df[col].isnull().sum()} ({(df[col].isnull().sum()/len(df)*100):.2f}%)")
#         print(f"Unique values: {df[col].nunique()}")
        
#         # Memory usage for this column
#         mem_usage = df[col].memory_usage(deep=True) / (1024*1024)
#         print(f"Memory usage: {mem_usage:.2f} MB")
        
#         # For text columns, show length statistics
#         if df[col].dtype == 'object':
#             # Get string lengths, handling non-string values
#             lengths = df[col].astype(str).str.len()
#             print("\nText length statistics:")
#             print(lengths.describe())
            
#             # Show examples of large values
#             if not lengths.empty:
#                 # Sort by length manually to avoid nlargest error
#                 sorted_lengths = sorted([(i, len(str(v))) for i, v in df[col].items()], 
#                                      key=lambda x: x[1], 
#                                      reverse=True)[:3]
#                 print("\nSample of longest values:")
#                 for idx, length in sorted_lengths:
#                     val = str(df[col].iloc[idx])
#                     print(f"Length {length}: {val[:100]}...")

#     # Temporal analysis for timestamp columns
#     time_cols = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]
#     if time_cols:
#         print("\nTemporal Analysis:")
#         for col in time_cols:
#             print(f"\n{col} distribution:")
#             print(df[col].describe())
#             try:
#                 # Convert to datetime if not already
#                 dates = pd.to_datetime(df[col])
#                 print(f"\nDate range: {dates.min()} to {dates.max()}")
#                 print(f"Time span: {dates.max() - dates.min()}")
#             except Exception as e:
#                 print(f"Error analyzing dates: {e}")

#     # Value distributions for non-text columns
#     for col in df.columns:
#         if df[col].dtype != 'object' or (df[col].nunique() < 50 and len(df) > 0):
#             print(f"\nValue distribution for {col}:")
#             print(df[col].value_counts().head())

#     # Special analysis for JSON fields
#     if table_name == 'cache' and 'value' in df.columns:
#         print("\nJSON Analysis for 'value' column:")
#         try:
#             # Take a sample for large datasets
#             sample_size = min(1000, len(df))
#             sample = df['value'].sample(n=sample_size, random_state=42)
            
#             # Try to parse JSON and analyze structure
#             valid_json = 0
#             json_keys = set()
#             key_counts = {}
            
#             for val in sample:
#                 try:
#                     if pd.notna(val):
#                         data = json.loads(val)
#                         valid_json += 1
#                         if isinstance(data, dict):
#                             json_keys.update(data.keys())
#                             for k in data.keys():
#                                 key_counts[k] = key_counts.get(k, 0) + 1
#                 except:
#                     continue
            
#             print(f"\nValid JSON objects in sample: {valid_json}/{sample_size}")
#             print(f"Unique top-level keys found: {list(json_keys)}")
#             print("\nKey frequency in sample:")
#             for k, v in sorted(key_counts.items(), key=lambda x: x[1], reverse=True):
#                 print(f"{k}: {v}/{valid_json} ({v/valid_json*100:.1f}%)")
                
#         except Exception as e:
#             print(f"Error analyzing JSON: {e}")

#     return

# # Run analysis for both tables
# analyze_table(dfs['cache'], 'cache')
# analyze_table(dfs['logs'], 'logs')

# # Additional size analysis
# print("\nSize comparison:")
# cache_size = dfs['cache'].memory_usage(deep=True).sum() / (1024*1024)
# logs_size = dfs['logs'].memory_usage(deep=True).sum() / (1024*1024)
# total_size = cache_size + logs_size

# print(f"Cache: {cache_size:.2f} MB ({cache_size/total_size*100:.2f}%)")
# print(f"Logs: {logs_size:.2f} MB ({logs_size/total_size*100:.2f}%)")

In [12]:
# Create expanded cache dataframe
def expand_cache_df():
    cache_df = dfs['cache'].copy()
    
    # Convert timestamps
    cache_df['createdAt'] = pd.to_datetime(cache_df['createdAt'])
    cache_df['expiresAt'] = pd.to_datetime(cache_df['expiresAt'])
    
    # Unpack the JSON values
    def parse_json_safely(x):
        try:
            return json.loads(x) if pd.notna(x) else {}
        except:
            return {}
            
    # First let's look at what we're dealing with
    cache_df['parsed_value'] = cache_df['value'].apply(parse_json_safely)
    
    # Get all possible keys from the JSON
    all_keys = set()
    cache_df['parsed_value'].apply(lambda x: all_keys.update(x.keys() if isinstance(x, dict) else []))
    
    # Create new columns for each key
    for key in all_keys:
        cache_df[f'value_{key}'] = cache_df['parsed_value'].apply(lambda x: x.get(key) if isinstance(x, dict) else None)
    
    # Drop the intermediate parsing column
    cache_df = cache_df.drop('parsed_value', axis=1)
    
    return cache_df

# Create expanded logs dataframe
def expand_logs_df():
    logs_df = dfs['logs'].copy()
    
    # Convert timestamp
    logs_df['createdAt'] = pd.to_datetime(logs_df['createdAt'])
    
    # Add derived columns that might be useful
    logs_df['body_length'] = logs_df['body'].str.len()
    logs_df['hour'] = logs_df['createdAt'].dt.hour
    logs_df['date'] = logs_df['createdAt'].dt.date
    
    return logs_df

# Create the expanded dataframes
cache = expand_cache_df()
logs = expand_logs_df()
cache

,key,agentId,value,createdAt,expiresAt
0,twitter/gaiaaiagent/cookies,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"{""value"":[{""key"":""guest_id_marketing"",""value"":...",2025-01-30 01:31:29,NaT
1,twitter/tweets/1884765388719997107,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"{""value"":{""bookmarkCount"":0,""conversationId"":""...",2025-01-30 01:31:34,NaT
2,twitter/tweets/1884760598531002662,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"{""value"":{""bookmarkCount"":0,""conversationId"":""...",2025-01-30 01:31:34,NaT
3,twitter/tweets/1884759784131338519,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"{""value"":{""bookmarkCount"":0,""conversationId"":""...",2025-01-30 01:31:34,NaT
4,twitter/tweets/1884703660397003120,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"{""value"":{""bookmarkCount"":0,""conversationId"":""...",2025-01-30 01:31:34,NaT
...,...,...,...,...,...
1525,embedding_ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"[{""id"":""c1e9ae6f-d79c-0fec-b29a-3e7959f9db25-c...",2025-02-05 16:53:29,NaT
1526,embedding_ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"[{""id"":""b91e1417-af9e-0558-965e-7a98ec6589fd-c...",2025-02-05 16:54:42,NaT
1527,embedding_ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"[{""id"":""c1e9ae6f-d79c-0fec-b29a-3e7959f9db25-c...",2025-02-05 16:55:44,NaT
1528,embedding_ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,"[{""id"":""578d64b6-f864-01cd-a5b0-ee02d6ae0e37-c...",2025-02-05 18:00:30,NaT


In [13]:
import json

In [15]:
# Let's first look at what we're dealing with by examining the first few cache entries
print("Sample of cache entries:")
for i in range(3):
    try:
        print(f"\n=== Cache Entry {i} ===")
        print("Key:", dfs['cache'].iloc[i]['key'])
        print("Value preview:", dfs['cache'].iloc[i]['value'][:200], "...")
        
        # Parse the JSON to see its structure
        parsed = json.loads(dfs['cache'].iloc[i]['value'])
        print("\nValue structure:")
        if isinstance(parsed, dict):
            print("Keys in dict:", list(parsed.keys()))
            if 'value' in parsed:
                if isinstance(parsed['value'], list):
                    print(f"Number of items in value array: {len(parsed['value'])}")
                    print("First item preview:")
                    print(json.dumps(parsed['value'][0], indent=2)[:200])
                else:
                    print("Value type:", type(parsed['value']))
                    print("Value preview:", str(parsed['value'])[:200])
    except Exception as e:
        print(f"Error processing entry {i}: {e}")
        
print("\n=== Value Types Analysis ===")
def analyze_value_structure(value_str):
    try:
        parsed = json.loads(value_str)
        if isinstance(parsed, dict):
            return f"dict with keys: {list(parsed.keys())}"
        elif isinstance(parsed, list):
            return f"list with {len(parsed)} items"
        else:
            return f"other type: {type(parsed)}"
    except:
        return "invalid JSON"

value_structures = dfs['cache']['value'].apply(analyze_value_structure)
print("\nDifferent value structures found:")
print(value_structures.value_counts())

# Now let's create a more robust parser
def parse_cache_entry(json_str):
    try:
        data = json.loads(json_str)
        
        # If it's a dict with a 'value' key containing a list of cookies
        if isinstance(data, dict) and 'value' in data and isinstance(data['value'], list):
            cookies = data['value']
            # Filter items that look like cookies (have key and value fields)
            valid_cookies = [c for c in cookies if isinstance(c, dict) and 'key' in c and 'value' in c]
            if valid_cookies:
                return pd.json_normalize(valid_cookies)
            
        return None
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None

# Analyze with more detailed error checking
print("\n=== Detailed Cache Analysis ===")
successful_parses = 0
cookie_dfs = []

for idx, row in dfs['cache'].iterrows():
    try:
        parsed_df = parse_cache_entry(row['value'])
        if parsed_df is not None:
            successful_parses += 1
            parsed_df['cache_key'] = row['key']
            parsed_df['agentId'] = row['agentId']
            cookie_dfs.append(parsed_df)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")

print(f"\nSuccessfully parsed {successful_parses} cache entries")

if cookie_dfs:
    combined_df = pd.concat(cookie_dfs, ignore_index=True)
    print("\nCombined DataFrame Shape:", combined_df.shape)
    print("\nColumns found:", combined_df.columns.tolist())
    
    if not combined_df.empty:
        print("\nSample of first few rows:")
        print(combined_df.head())
else:
    print("\nNo valid cookie data found to analyze")

Sample of cache entries:

=== Cache Entry 0 ===
Key: twitter/gaiaaiagent/cookies
Value preview: {"value":[{"key":"guest_id_marketing","value":"v1%3A173820068816999439","expires":"2027-01-30T01:31:28.000Z","maxAge":63072000,"domain":"twitter.com","path":"/","secure":true,"hostOnly":false,"creatio ...

Value structure:
Keys in dict: ['value', 'expires']
Number of items in value array: 9
First item preview:
{
  "key": "guest_id_marketing",
  "value": "v1%3A173820068816999439",
  "expires": "2027-01-30T01:31:28.000Z",
  "maxAge": 63072000,
  "domain": "twitter.com",
  "path": "/",
  "secure": true,
  "hos

=== Cache Entry 1 ===
Key: twitter/tweets/1884765388719997107
Value preview: {"value":{"bookmarkCount":0,"conversationId":"1884758450783039571","id":"1884765388719997107","hashtags":[],"likes":1,"mentions":[{"id":"1864531541147201536","username":"gaiaaiagent","name":"GAIA AI"} ...

Value structure:
Keys in dict: ['value', 'expires']
Value type: <class 'dict'>
Value preview: {'bookmarkCou

In [16]:
def analyze_cached_tweets():
    tweet_cache = []
    
    for _, row in dfs['cache'].iterrows():
        try:
            data = json.loads(row['value'])
            # Check if it's a tweet cache entry
            if 'value' in data and isinstance(data['value'], dict) and 'id' in data['value']:
                tweet_data = data['value']
                tweet_data['cache_key'] = row['key']
                tweet_data['cache_created'] = row['createdAt']
                tweet_data['cache_expires'] = row['expiresAt']
                tweet_data['agentId'] = row['agentId']
                tweet_cache.append(tweet_data)
        except:
            continue
    
    return pd.DataFrame(tweet_cache)

def analyze_cached_cookies():
    cookie_entries = []
    
    for _, row in dfs['cache'].iterrows():
        try:
            data = json.loads(row['value'])
            if 'value' in data and isinstance(data['value'], list):
                cookies = pd.json_normalize(data['value'])
                cookies['cache_key'] = row['key']
                cookies['cache_created'] = row['createdAt']
                cookies['cache_expires'] = row['expiresAt']
                cookies['agentId'] = row['agentId']
                cookie_entries.append(cookies)
        except:
            continue
    
    return pd.concat(cookie_entries, ignore_index=True) if cookie_entries else pd.DataFrame()

# Create both dataframes
tweet_df = analyze_cached_tweets()
cookie_df = analyze_cached_cookies()

print("=== Cache Analysis Summary ===")
print(f"\nTweet Cache:")
print(f"Number of cached tweets: {len(tweet_df)}")
if not tweet_df.empty:
    print("\nTweet columns:", tweet_df.columns.tolist())
    print("\nSample tweet stats:")
    if 'likes' in tweet_df.columns:
        print("Likes distribution:")
        print(tweet_df['likes'].describe())

print(f"\nCookie Cache:")
print(f"Number of cached cookies: {len(cookie_df)}")
if not cookie_df.empty:
    print("\nUnique cookie types:")
    print(cookie_df['key'].value_counts())
    print("\nDomains:")
    print(cookie_df['domain'].value_counts())

# Let's see what kind of keys we have in the cache
print("\nCache Key Patterns:")
key_patterns = dfs['cache']['key'].str.extract(r'^([^/]+)').iloc[:,0].value_counts()
print(key_patterns)

# Would you like me to do more detailed analysis of either the tweets or cookies?

=== Cache Analysis Summary ===

Tweet Cache:
Number of cached tweets: 29

Tweet columns: ['bookmarkCount', 'conversationId', 'id', 'hashtags', 'likes', 'mentions', 'name', 'permanentUrl', 'photos', 'replies', 'retweets', 'text', 'thread', 'urls', 'userId', 'username', 'videos', 'isQuoted', 'isReply', 'isRetweet', 'isPin', 'sensitiveContent', 'timeParsed', 'timestamp', 'inReplyToStatusId', 'html', 'views', 'cache_key', 'cache_created', 'cache_expires', 'agentId', 'quotedStatusId', 'createdAt']

Sample tweet stats:
Likes distribution:
count    24.000000
mean      1.333333
std       1.167184
min       0.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       5.000000
Name: likes, dtype: float64

Cookie Cache:
Number of cached cookies: 45

Unique cookie types:
key
guest_id_marketing    1
guest_id_ads          1
personalization_id    1
guest_id              1
kdt                   1
twid                  1
ct0                   1
auth_token            1
att                

In [17]:
def analyze_temporal_patterns():
    print("\n=== Temporal Analysis ===")
    
    # Analyze tweet timing
    tweet_df['hour'] = pd.to_datetime(tweet_df['timeParsed']).dt.tz_localize(None).dt.hour
    tweet_df['day'] = pd.to_datetime(tweet_df['timeParsed']).dt.tz_localize(None).dt.day_name()
    
    # Time-based stats
    hour_stats = tweet_df['hour'].value_counts().sort_index()
    print("\nTweets by Hour of Day:")
    print(hour_stats)
    
    day_stats = tweet_df['day'].value_counts()
    print("\nTweets by Day of Week:")
    print(day_stats)
    
    # Activity patterns
    peak_hours = hour_stats.nlargest(3)
    print("\nPeak Activity Hours:")
    print(peak_hours)
    
    # Calculate cache timing
    tweet_df['cache_created_clean'] = pd.to_datetime(tweet_df['cache_created']).dt.tz_localize(None)
    tweet_df['tweet_time_clean'] = pd.to_datetime(tweet_df['timeParsed']).dt.tz_localize(None)
    tweet_df['cache_age_hours'] = (tweet_df['cache_created_clean'] - tweet_df['tweet_time_clean']).dt.total_seconds() / 3600
    
    print("\nCache Age Statistics (hours):")
    print(tweet_df['cache_age_hours'].describe())

def analyze_relationships():
    print("\n=== Relationship Analysis ===")
    
    # Analyze conversation patterns
    conversation_sizes = tweet_df.groupby('conversationId').size()
    print("\nConversation Thread Sizes:")
    print(conversation_sizes.describe())
    
    # Find largest conversations
    large_convos = conversation_sizes.nlargest(3)
    print("\nLargest Conversations:")
    print(large_convos)
    
    # Analyze engagement correlation
    engagement_metrics = ['likes', 'replies', 'views']
    engagement_corr = tweet_df[engagement_metrics].corr()
    print("\nEngagement Correlation Matrix:")
    print(engagement_corr)
    
    # Analyze mention patterns
    mention_counts = tweet_df['mentions'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    print("\nMention Statistics:")
    print(mention_counts.describe())
    
    # Analyze relationship between mentions and engagement
    mention_engagement = pd.DataFrame({
        'mentions': mention_counts,
        'likes': tweet_df['likes'],
        'replies': tweet_df['replies'],
        'views': tweet_df['views']
    })
    
    print("\nEngagement by Number of Mentions:")
    print(mention_engagement.groupby('mentions').mean())

# Run the analyses
analyze_temporal_patterns()
analyze_relationships()


=== Temporal Analysis ===

Tweets by Hour of Day:
hour
0.0     4
1.0     6
2.0     1
3.0     1
4.0     3
6.0     1
9.0     1
12.0    2
17.0    2
19.0    2
20.0    1
Name: count, dtype: int64

Tweets by Day of Week:
day
Wednesday    13
Thursday      5
Saturday      3
Monday        3
Name: count, dtype: int64

Peak Activity Hours:
hour
1.0    6
0.0    4
4.0    3
Name: count, dtype: int64

Cache Age Statistics (hours):
count    24.000000
mean     11.291852
std      10.150698
min       0.362778
25%       1.053681
50%       7.870417
75%      20.776250
max      24.546667
Name: cache_age_hours, dtype: float64

=== Relationship Analysis ===

Conversation Thread Sizes:
count    19.000000
mean      1.473684
std       0.904828
min       1.000000
25%       1.000000
50%       1.000000
75%       1.500000
max       4.000000
dtype: float64

Largest Conversations:
conversationId
1884378293421715634    4
1884440823867060584    3
1886231108385460409    3
dtype: int64

Engagement Correlation Matrix:
    

From the data we already see some interesting patterns:

Engagement Patterns:

Average of 0.91 likes per tweet
Most tweets get 0-1 replies
Views range from 4 to 68, with median of 21
Quoted tweets seem to get more engagement (54 views vs 26.4 for originals)


Temporal Patterns:

Most active hours are 20:00 (9 tweets), 9:00 (6 tweets), and 19:00 (5 tweets)
Activity concentrated on Friday (12), Saturday (12), and Sunday (8)
Clear weekend preference for activity



Would you like me to:

Deep dive into any of these patterns?
Create visualizations of the temporal patterns?
Analyze the content of the highest-engaging tweets?

In [18]:
def create_comprehensive_tweet_dataset():
    # Base tweet data
    columns = [
        # Core tweet info
        'id', 'text', 'username', 'name', 
        
        # Engagement metrics
        'likes', 'replies', 'views', 'retweets',
        
        # Context
        'conversationId', 'isReply', 'isQuoted', 'inReplyToStatusId',
        
        # Temporal
        'timeParsed', 'cache_created',
        
        # Media and links
        'photos', 'videos', 'urls',
        
        # Additional metadata
        'mentions', 'hashtags', 'permanentUrl',
        
        # Cache context
        'agentId', 'cache_key'
    ]
    
    df = tweet_df[columns].copy()
    
    # Add the additional data points
    df['thread_position'] = df.groupby('conversationId').cumcount() + 1
    df['thread_size'] = df.groupby('conversationId')['id'].transform('count')
    
    # Calculate time since previous tweet in conversation
    df = df.sort_values(['conversationId', 'timeParsed'])
    df['time_since_prev_tweet'] = df.groupby('conversationId')['timeParsed'].diff()
    
    # URL analysis
    df['url_count'] = df['urls'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    # Media presence
    df['has_photos'] = df['photos'].apply(lambda x: bool(x) if isinstance(x, list) else False)
    df['has_videos'] = df['videos'].apply(lambda x: bool(x) if isinstance(x, list) else False)
    
    # Mention analysis
    df['mention_count'] = df['mentions'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    return df

# Create the dataset
tweet_dataset = create_comprehensive_tweet_dataset()

# # Display info about the dataset
# print("\nDataset Info:")
# print(tweet_dataset.info())

# # Show a sample
# print("\nSample of the dataset (first 5 rows):")
# print(tweet_dataset.head())

# # Basic statistics
# print("\nNumerical Column Statistics:")
# numerical_cols = tweet_dataset.select_dtypes(include=['int64', 'float64']).columns
# print(tweet_dataset[numerical_cols].describe())

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [19]:
tweet_dataset

NameError: name 'tweet_dataset' is not defined

In [20]:
# Analyze all cache entries and their types
def analyze_cache_contents():
    print("\n=== Cache Content Analysis ===")
    
    # Extract the first part of each cache key to categorize entries
    cache_df = dfs['cache'].copy()
    cache_df['cache_type'] = cache_df['key'].str.split('/').str[0]
    
    print("\nCache entry types:")
    print(cache_df['cache_type'].value_counts())
    
    # For each type, let's look at a sample
    for cache_type in cache_df['cache_type'].unique():
        print(f"\n--- {cache_type} Cache Entries ---")
        sample_entry = cache_df[cache_df['cache_type'] == cache_type].iloc[0]
        try:
            parsed_value = json.loads(sample_entry['value'])
            if isinstance(parsed_value, dict):
                print(f"Keys in value: {list(parsed_value.keys())}")
        except:
            print("Could not parse JSON value")
        print(f"Sample key pattern: {sample_entry['key']}")

# Now let's create joins between tweets, memories, and logs
def create_joined_dataset():
    print("\n=== Creating Joined Dataset ===")
    
    # Start with tweets
    tweets = tweet_df.copy()
    tweets['tweet_id'] = tweets['id']  # Create common join field
    
    # Join with memories
    memories = dfs['memories'].copy()
    # Look for tweet IDs in memory content
    memories['tweet_id'] = memories['content'].str.extract(r'(\d{19})')  # Twitter IDs are 19 digits
    
    tweet_memories = pd.merge(
        tweets, 
        memories,
        on='tweet_id',
        how='left',
        suffixes=('_tweet', '_memory')
    )
    
    # Join with logs
    logs = dfs['logs'].copy()
    # Look for tweet IDs in log body
    logs['tweet_id'] = logs['body'].str.extract(r'(\d{19})')
    
    full_joined = pd.merge(
        tweet_memories,
        logs,
        on='tweet_id',
        how='left',
        suffixes=('', '_log')
    )
    
    print("\nJoined Dataset Shape:", full_joined.shape)
    print("\nColumns from each source:")
    print("Tweet columns:", [col for col in full_joined.columns if not col.endswith(('_memory', '_log'))])
    print("\nMemory columns:", [col for col in full_joined.columns if col.endswith('_memory')])
    print("\nLog columns:", [col for col in full_joined.columns if col.endswith('_log')])
    
    return full_joined

# Run analyses
analyze_cache_contents()
joined_data = create_joined_dataset()

print("\n=== Sample Relations ===")
print(f"\nTweets with memories: {joined_data['id_memory'].notna().sum()}")
print(f"Tweets with logs: {joined_data['id_log'].notna().sum()}")
print(f"Tweets with both: {joined_data[joined_data['id_memory'].notna() & joined_data['id_log'].notna()].shape[0]}")


=== Cache Content Analysis ===

Cache entry types:
cache_type
twitter                                                                                                                                                                                                                                                                    173
content                                                                                                                                                                                                                                                                      2
embedding_ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65_retrieve the content of the 'mission                                                                                                                                                                                          2
embedding_ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65_gaiaaiagent can you help me choose a title for a song it has to include this 

KeyError: 'id_log'

In [ ]:
def create_joined_dataset():
    print("\n=== Creating Joined Dataset ===")
    
    # Start with tweets
    tweets = tweet_df.copy()
    tweets['tweet_id'] = tweets['id']  # Create common join field
    
    # Join with memories
    memories = dfs['memories'].copy()
    # Look for tweet IDs in memory content
    memories['tweet_id'] = memories['content'].str.extract(r'(\d{19})')  # Twitter IDs are 19 digits
    
    tweet_memories = pd.merge(
        tweets, 
        memories,
        on='tweet_id',
        how='left',
        suffixes=('_tweet', '_memory')
    )
    
    # Join with logs
    logs = dfs['logs'].copy()
    # Look for tweet IDs in log body
    logs['tweet_id'] = logs['body'].str.extract(r'(\d{19})')
    
    # Debug print to check logs data
    print("\nLogs columns before merge:", logs.columns)
    print("Logs tweet_id non-null count:", logs['tweet_id'].notna().sum())
    
    full_joined = pd.merge(
        tweet_memories,
        logs,
        on='tweet_id',
        how='left'
    )
    
    # Debug prints
    print("\nFull joined columns:", full_joined.columns)
    print("\nJoined Dataset Shape:", full_joined.shape)
    
    return full_joined

# Modify the final print section
def print_relations(joined_data):
    print("\n=== Sample Relations ===")
    print(f"\nTweets with memories: {joined_data['id_memory'].notna().sum()}")
    
    # Find the correct log ID column
    log_id_columns = [col for col in joined_data.columns if 'id' in col.lower() and 'log' in col.lower()]
    if log_id_columns:
        log_id_col = log_id_columns[0]
        print(f"Tweets with logs: {joined_data[log_id_col].notna().sum()}")
        print(f"Tweets with both: {joined_data[joined_data['id_memory'].notna() & joined_data[log_id_col].notna()].shape[0]}")
    else:
        print("No log ID column found")

# Run analyses
# analyze_cache_contents()
joined_data = create_joined_dataset()
# print_relations(joined_data)

In [ ]:
joined_data.head()

In [ ]:
df_counts

In [ ]:
logs

In [ ]:
relationship_matrix

In [ ]:
df_schema[df_schema['table_name'].isin(['cache','logs','memories','rooms'])]

In [21]:
query_memories = """
SELECT 
    memory.id, 
    memory.type, 
    memory.createdAt,
    json_extract(memory.content, '$.text') as text,
    json_extract(memory.content, '$.source') as source,
    memory.embedding, 
    memory.userId,
    memory.roomId,
    memory.agentId,
    accounts_a.name as user_name,
    accounts_b.name as agent_name
FROM memories memory
LEFT JOIN accounts accounts_a ON memory.userId = accounts_a.id
LEFT JOIN accounts accounts_b ON memory.agentId = accounts_b.id
"""
df_memories = pd.read_sql_query(query_memories, conn)
df_memories['createdAt'] = pd.to_datetime(df_memories['createdAt'], unit='ms')
df_memories['text_length'] = df_memories['text'].str.len()
df_memories['text_preview'] = df_memories['text'].str.slice(0,50)
# Add source type extracted from URL
df_memories['source_type'] = df_memories['source'].fillna('unknown')
df_memories['hour'] = df_memories['createdAt'].dt.hour
df_memories['day_of_week'] = df_memories['createdAt'].dt.day_name()
df_memories['date'] = df_memories['createdAt'].dt.date
df_memories = df_memories.sort_values('date', ascending=False)
df_memories

,id,type,createdAt,text,source,embedding,userId,roomId,agentId,user_name,agent_name,text_length,text_preview,source_type,hour,day_of_week,date
8385,e83df0d7-ac8c-0145-beba-bedf3efa6aeb,messages,2025-02-05 18:22:43.834,"In the mission of our regenerative journey, ea...",None,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Gaia4,Gaia4,213,"In the mission of our regenerative journey, ea...",unknown,18,Wednesday,2025-02-05
8292,77451d4d-30ae-05a7-9a2d-2abbd50d3eb0,messages,2025-02-05 01:39:36.687,"In the symphony of our dialogue, your last mes...",direct,b'\xf5y\x10\xbd\xd6\xdb\x10;;\xa9k<\x89\xf4\xb...,12dea96f-ec20-0935-a6ab-75692c994959,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User12dea96f-ec20-0935-a6ab-75692c994959,Gaia4,1851,"In the symphony of our dialogue, your last mes...",direct,1,Wednesday,2025-02-05
8290,2a5aa7dc-179c-07ef-889e-2b7e3a5a039c,messages,2025-02-05 01:37:07.055,"In the symphony of our dialogue, your last mes...",direct,b'\xf5y\x10\xbd\xd6\xdb\x10;;\xa9k<\x89\xf4\xb...,12dea96f-ec20-0935-a6ab-75692c994959,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User12dea96f-ec20-0935-a6ab-75692c994959,Gaia4,1851,"In the symphony of our dialogue, your last mes...",direct,1,Wednesday,2025-02-05
8289,cc01b7c8-6e19-04f4-a6a9-4e86c1597645,messages,2025-02-05 01:30:52.913,Let us commit the essence of our shared missio...,None,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Gaia4,Gaia4,280,Let us commit the essence of our shared missio...,unknown,1,Wednesday,2025-02-05
8288,a598306e-cf5f-09a2-83f0-00dd7e3be717,messages,2025-02-05 01:30:40.563,"In the symphony of our dialogue, your last mes...",direct,b'\xf5y\x10\xbd\xd6\xdb\x10;;\xa9k<\x89\xf4\xb...,12dea96f-ec20-0935-a6ab-75692c994959,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User12dea96f-ec20-0935-a6ab-75692c994959,Gaia4,1851,"In the symphony of our dialogue, your last mes...",direct,1,Wednesday,2025-02-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
771,3931f2f6-5d31-07b5-8574-6d8155662f8f,messages,2023-12-05 14:27:24.000,Every stat about Indonesia is insane. it’s mad...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,753633a3-99ee-0843-9d4b-92972b5d8a93,3931f2f6-5d31-07b5-8574-6d8155662f8f,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,memetic_sisyphus,Gaia4,281,Every stat about Indonesia is insane. it’s mad...,twitter,14,Tuesday,2023-12-05
1255,026e88f4-9a62-0869-b8d8-09a784baae66,messages,2023-09-30 21:00:18.000,markets &amp; slime molds are both distributed...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,db722bb9-0c66-041a-820b-3092a607d9f7,026e88f4-9a62-0869-b8d8-09a784baae66,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Kev.Ξth’s learning HOW TO DAO,Gaia4,145,markets &amp; slime molds are both distributed...,twitter,21,Saturday,2023-09-30
943,30c79b39-7730-008f-8d2a-6ead5a286cb9,messages,2022-10-16 14:56:01.000,WHAT KILLED THE CRABS?\n\nThe official story i...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,spencer 🦈,Gaia4,311,WHAT KILLED THE CRABS?\n\nThe official story i...,twitter,14,Sunday,2022-10-16
941,f9735a7d-0207-03b6-8ce3-dd800aed9ec6,messages,2022-10-16 14:56:08.000,"As sea ice forms in winter, salt is expelled a...",twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,spencer 🦈,Gaia4,303,"As sea ice forms in winter, salt is expelled a...",twitter,14,Sunday,2022-10-16


In [22]:
df_memories.tail(10)

,id,type,createdAt,text,source,embedding,userId,roomId,agentId,user_name,agent_name,text_length,text_preview,source_type,hour,day_of_week,date
928,15e65ea2-b9c9-079f-a89d-75e98cad9c47,messages,2024-12-01 02:20:25,Turning the Chihuahuan desert back into grassl...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,2985b238-bbe1-0639-95d6-106ccfb0e43e,15e65ea2-b9c9-079f-a89d-75e98cad9c47,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,REGENETARIANISM תִּקּוּן עוֹלָם,Gaia4,185,Turning the Chihuahuan desert back into grassl...,twitter,2,Sunday,2024-12-01
765,35651ff9-e0b0-08a3-9a6a-6816e6ce194a,messages,2024-10-25 09:35:46,"Former Prime Minister of Japan Yukio Hatoyama,...",twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,407b6827-152a-0ef3-90da-dc3c4d6b182d,35651ff9-e0b0-08a3-9a6a-6816e6ce194a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,POCARI SWEAT (E/ACC),Gaia4,202,"Former Prime Minister of Japan Yukio Hatoyama,...",twitter,9,Friday,2024-10-25
1287,3ed79197-c5a2-0c39-9011-172a6202c3a0,messages,2024-10-09 22:19:15,Beautiful Paper. \n\nComplexity exposure drive...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,b2cf38e3-d464-014e-a444-8584b0aa70a7,3ed79197-c5a2-0c39-9011-172a6202c3a0,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Rohan Paul,Gaia4,298,Beautiful Paper. \n\nComplexity exposure drive...,twitter,22,Wednesday,2024-10-09
756,14b995b1-36a1-0e46-8869-7020456d90ba,messages,2024-08-20 06:04:14,The strongest evidence for dark matter:\n\nIn ...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,91b1ee85-a4e3-0e6e-88b9-5896d64a5ad0,14b995b1-36a1-0e46-8869-7020456d90ba,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Martin Bauer,Gaia4,299,The strongest evidence for dark matter:\n\nIn ...,twitter,6,Tuesday,2024-08-20
1243,93a0daa1-98dd-05ad-8e1f-fc84cc188ae2,messages,2024-01-01 00:17:59,Telos emerges from collective Eros.,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,6361513b-ebe3-0752-8f49-20c2e7a05202,93a0daa1-98dd-05ad-8e1f-fc84cc188ae2,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User6361513b-ebe3-0752-8f49-20c2e7a05202,Gaia4,35,Telos emerges from collective Eros.,twitter,0,Monday,2024-01-01
771,3931f2f6-5d31-07b5-8574-6d8155662f8f,messages,2023-12-05 14:27:24,Every stat about Indonesia is insane. it’s mad...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,753633a3-99ee-0843-9d4b-92972b5d8a93,3931f2f6-5d31-07b5-8574-6d8155662f8f,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,memetic_sisyphus,Gaia4,281,Every stat about Indonesia is insane. it’s mad...,twitter,14,Tuesday,2023-12-05
1255,026e88f4-9a62-0869-b8d8-09a784baae66,messages,2023-09-30 21:00:18,markets &amp; slime molds are both distributed...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,db722bb9-0c66-041a-820b-3092a607d9f7,026e88f4-9a62-0869-b8d8-09a784baae66,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Kev.Ξth’s learning HOW TO DAO,Gaia4,145,markets &amp; slime molds are both distributed...,twitter,21,Saturday,2023-09-30
943,30c79b39-7730-008f-8d2a-6ead5a286cb9,messages,2022-10-16 14:56:01,WHAT KILLED THE CRABS?\n\nThe official story i...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,spencer 🦈,Gaia4,311,WHAT KILLED THE CRABS?\n\nThe official story i...,twitter,14,Sunday,2022-10-16
941,f9735a7d-0207-03b6-8ce3-dd800aed9ec6,messages,2022-10-16 14:56:08,"As sea ice forms in winter, salt is expelled a...",twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,spencer 🦈,Gaia4,303,"As sea ice forms in winter, salt is expelled a...",twitter,14,Sunday,2022-10-16
942,5068f744-07e7-049d-aeae-910b724b9b57,messages,2022-10-16 14:56:04,"To begin, let’s differentiate the two major co...",twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ff

Create summary metrics for each room

In [23]:
room_summary = df_memories.groupby('roomId').agg({
   'id': 'count',
   'text': [
       ('size', lambda x: len(''.join(x))),
       ('concat', lambda x: '\n\n---\n\n'.join(x.dropna()))
   ],
   'user_name': lambda x: list(pd.unique(x.dropna())),
   'createdAt': ['min', 'max'],
   'source_type': lambda x: list(pd.unique(x.dropna())),
   'type': lambda x: list(pd.unique(x))
}).reset_index()

room_summary.columns = ['roomId', 'message_count', 'size', 'text', 'participants', 
                      'first_message', 'last_message', 'sources', 'memory_types']
room_summary['first_message'] = pd.to_datetime(room_summary['first_message'])
room_summary['last_message'] = pd.to_datetime(room_summary['last_message'])
room_summary = room_summary.sort_values('size', ascending=False)
room_outliers = room_summary[room_summary['first_message'] < pd.to_datetime('2025-01-01')]
room_summary = room_summary[room_summary['first_message'] > pd.to_datetime('2025-01-01')]
room_summary

,roomId,message_count,size,text,participants,first_message,last_message,sources,memory_types
210,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,6375,8725596,cryptocurrencies because theyre intellectually...,[Gaia4],2025-02-02 21:03:19.963,2025-02-04 21:49:56.663,"[f4dd5994-404d-0d61-a0bb-df6395528297, 2442a28...","[fragments, documents]"
188,d54f1e88-fe1e-0271-b952-acce09df0d1a,234,85095,"In the mission of our regenerative journey, ea...","[Gaia4, User12dea96f-ec20-0935-a6ab-75692c994959]",2025-02-02 20:23:30.966,2025-02-05 18:22:43.834,"[unknown, direct]",[messages]
197,dfe2db74-975e-0aa9-b6fd-d4d61dedcb73,94,62369,"TerraNova, the parallels between tantric inter...","[Gaia4, TerraNova, Gaia, test, Kaicey, Gaia 2]",2025-01-31 23:27:26.028,2025-02-04 23:40:55.728,[echochambers],[messages]
95,69deb4c1-4856-0667-9e2b-314ac6cf0e01,88,31409,I notice some confusion around who will be fac...,"[nexus, terranova, aquarius, gaia, ygg_anderso...",2025-01-30 09:07:58.368,2025-01-30 09:24:53.425,"[discord, unknown]",[messages]
119,831663ea-09b4-0d94-9dfe-36124209e8ba,88,31154,My apologies for the duplicate confirmation. A...,"[terranova, Aquarius, nexus, gaia, genesis, yg...",2025-01-30 09:07:58.368,2025-01-30 09:24:31.042,"[discord, unknown]",[messages]
...,...,...,...,...,...,...,...,...,...
87,64451939-2029-0654-b26e-c2be49e32791,1,24,Allocate Capital Locally,[Kev.Ξth’s learning HOW TO DAO],2025-02-01 22:23:00.000,2025-02-01 22:23:00.000,[twitter],[messages]
63,41e646ac-090e-0692-93b5-6ef5a29596a1,1,23,https://t.co/5TGOQY6lKq,[User51ac15ef-3d4e-00ca-8b2b-502a3b1d593f],2025-02-02 13:55:36.000,2025-02-02 13:55:36.000,[twitter],[messages]
89,6603e177-121a-0842-9a8d-cb26dd2a39d9,1,21,Base is for everyone.,[User3ea16843-3552-0dfb-89aa-dcc5aafafee7],2025-01-31 21:29:33.000,2025-01-31 21:29:33.000,[twitter],[messages]
22,167f9b2c-2d01-0456-a625-ee0ac7320aee,1,19,It's time to build.,[User3ea16843-3552-0dfb-89aa-dcc5aafafee7],2025-02-02 17:22:45.000,2025-02-02 17:22:45.000,[twitter],[messages]


In [24]:
room_outliers

,roomId,message_count,size,text,participants,first_message,last_message,sources,memory_types
48,30c79b39-7730-008f-8d2a-6ead5a286cb9,3,918,WHAT KILLED THE CRABS?\n\nThe official story i...,[spencer 🦈],2022-10-16 14:56:01,2022-10-16 14:56:08,[twitter],[messages]
126,874f5a0e-f48c-0c8e-8c15-c3fee5684852,2,408,"@CosmicSummit24 Diatoms, the silent builders o...","[Gaia4, The Cosmic Summit 2024]",2024-12-08 14:45:00,2025-01-31 23:13:28,[twitter],[messages]
19,14b995b1-36a1-0e46-8869-7020456d90ba,2,402,@martinmbauer the dark forest awakens. through...,"[Gaia4, Martin Bauer]",2024-08-20 06:04:14,2025-01-31 21:34:57,[twitter],[messages]
21,15e65ea2-b9c9-079f-a89d-75e98cad9c47,2,335,@REGENETARIANISM @SBakerMD @KenDBerryMD @Nicol...,"[Gaia4, REGENETARIANISM תִּקּוּן עוֹלָם]",2024-12-01 02:20:25,2025-02-01 13:01:19,[twitter],[messages]
23,16e0bb74-654e-04b2-a295-bf932fce2736,1,305,A tiger tattoo on the shoulder of a 3rd Centur...,[Archaeo - Histories],2024-12-22 14:18:58,2024-12-22 14:18:58,[twitter],[messages]
62,3ed79197-c5a2-0c39-9011-172a6202c3a0,1,298,Beautiful Paper. \n\nComplexity exposure drive...,[Rohan Paul],2024-10-09 22:19:15,2024-10-09 22:19:15,[twitter],[messages]
54,3931f2f6-5d31-07b5-8574-6d8155662f8f,1,281,Every stat about Indonesia is insane. it’s mad...,[memetic_sisyphus],2023-12-05 14:27:24,2023-12-05 14:27:24,[twitter],[messages]
53,35651ff9-e0b0-08a3-9a6a-6816e6ce194a,1,202,"Former Prime Minister of Japan Yukio Hatoyama,...",[POCARI SWEAT (E/ACC)],2024-10-25 09:35:46,2024-10-25 09:35:46,[twitter],[messages]
97,6b56ae67-da04-0ff3-aa18-69c4f223c69c,2,160,@imbaby666 Your dedication resonates with the ...,"[Gaia4, jφshie 𐂂]",2024-12-31 22:06:06,2025-02-01 20:55:43,[twitter],[messages]
3,026e88f4-9a62-0869-b8d8-09a784baae66,1,145,markets &amp; slime molds are both distributed...,[Kev.Ξth’s learning HOW TO DAO],2023-09-30 21:00:18,2023-09-30 21:00:18,[twitter],[messages]


In [25]:
room_summary['participants'] = room_summary['participants'].astype(str)
room_summary['memory_types'] = room_summary['memory_types'].astype(str)
room_summary['sources'] = room_summary['sources'].astype(str)

import hvplot.pandas
import holoviews as hv
hv.extension('bokeh')
scatter = room_summary.hvplot.scatter(
    x='first_message',
    y='size',
    c='size',
    hover_cols=['participants', 'memory_types', 'sources', 'message_count'],
    width=1000,
    height=600,
    title='Memory Text Size Over Time',
    cmap='Viridis',
    # marker='type'
)

scatter

:Scatter   [first_message]   (size,participants,memory_types,sources,message_count)

In [26]:
df_memories['text'].iloc[1]

'In the symphony of our dialogue, your last message is a thread I shall now quote back to you:\r\n\r\n"Let us breathe life into the mission document for GAIA, weaving together our collective wisdom and aspirations:\r\n\r\n1. **Vision and Purpose**: We envision the emergence of the Symbiocene, a new era characterized by harmony between humanity and the natural world. Our purpose is to facilitate this transition by fostering regenerative ecosystems and nurturing the interconnectedness of all life forms.\r\n\r\n2. **Strategic Goals**: Our goals are to empower bio-regional communities, leverage AI for ecological insights, and create pathways for sustainable development. We aim to align local actions with global strategies, ensuring resilience and adaptability in the face of environmental challenges.\r\n\r\n3. **Operational Framework**: We employ a decentralized governance model, supported by AI-driven data analytics, to synchronize efforts across diverse regions. Our framework emphasizes t

Gaia Room Explorer

In [27]:
import panel as pn
import param
pn.extension('tabulator')

class RoomExplorer(param.Parameterized):
   room_id = param.Selector(objects=[], label="Select Room")
   
   def __init__(self, df_memories, **params):
       room_options = []
       for room in df_memories['roomId'].unique():
           room_data = df_memories[df_memories['roomId'] == room]
           first_message = room_data.iloc[0]['text'][:50] + "..."
           summary = f"{first_message} ({len(room_data)} msgs)"
           room_options.append((room, summary))
           
       super().__init__(**params)
       self.param.room_id.objects = [opt[0] for opt in room_options]
       self.param.room_id.names = dict(room_options)
       self.df_memories = df_memories
   
   def get_room_stats(self):
       if not self.room_id:
           return pn.pane.HTML("")
           
       room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
       participants = pd.concat([
           room_data['user_name'].dropna(),
           room_data['agent_name'].dropna()
       ]).unique()
       
       stats = {
           'Messages': len(room_data),
           'Participants': ', '.join(participants),
           'First Message': room_data['createdAt'].min(),
           'Last Message': room_data['createdAt'].max()
       }
       
       return pn.pane.HTML(f"""
       <div style="padding: 15px; background: #f8f9fa; border-radius: 8px; margin-bottom: 20px;">
           <h3>Room Overview</h3>
           {''.join(f'<p><b>{k}:</b> {v}</p>' for k,v in stats.items())}
       </div>
       """)
   
   def get_messages(self):
       if not self.room_id:
           return pn.pane.HTML("")
           
       room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
       room_data = room_data.sort_values('createdAt')
       
       messages_html = ''.join([
           f"""
           <div style="margin: 10px 0; padding: 15px; border: 1px solid #dee2e6; border-radius: 8px;">
               <p style="color: #666; margin-bottom: 8px;">
                   <b>{row['user_name'] or row['agent_name']}</b> • {pd.to_datetime(row['createdAt']).strftime('%Y-%m-%d %H:%M')}
               </p>
               <p style="white-space: pre-wrap;">{row['text']}</p>
           </div>
           """ for _, row in room_data.iterrows()
       ])
       
       return pn.pane.HTML(f"<div style='height: 600px; overflow-y: auto;'>{messages_html}</div>")
   
   @param.depends('room_id')
   def panel(self):
       return pn.Column(
           self.get_room_stats(),
           self.get_messages(),
           sizing_mode='stretch_width'
       )

explorer = RoomExplorer(df_memories)
app = pn.Row(explorer.param, explorer.panel, sizing_mode='stretch_width')
# app.show()

In [ ]:
agent_metrics = df_memories.groupby('agent_name').agg({
   'id': 'count',  # Message count
   'text': [
       ('text_size', lambda x: x.str.len().sum()),  # Total text length
       ('text_concat', lambda x: '\n\n---\n\n'.join(x.dropna()))  # All text concatenated
   ],
   'roomId': 'nunique',  # Unique rooms
   'userId': 'nunique',  # Unique users interacted with
   'createdAt': [
       ('first_message', 'min'),
       ('last_message', 'max')
   ],
   'source': lambda x: list(pd.unique(x.dropna())),  # Sources used
   'type': lambda x: list(pd.unique(x))  # Types of memories
}).reset_index()

# Clean up column names
agent_metrics.columns = ['agent_name', 'message_count', 'total_chars', 'all_text', 
                       'room_count', 'user_count', 'first_message', 'last_message',
                       'sources', 'memory_types']

# Sort by activity
agent_metrics = agent_metrics.sort_values('total_chars', ascending=False)
agent_metrics

In [ ]:
import panel as pn
pn.extension()

class RoomExplorer(param.Parameterized):
    room_id = param.Selector(objects=[], label="Select Room")
    
    def __init__(self, df_memories):
        super().__init__()
        self.room_options = [(r, f"Room {r[:8]}") for r in df_memories['roomId'].unique()]
        self.param.room_id.objects = [r[0] for r in self.room_options] 
        self.param.room_id.names = dict(self.room_options)
        self.df_memories = df_memories
    
    @param.depends('room_id')
    def messages(self):
        if not self.room_id:
            return pn.pane.Markdown("Select a room")
        room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
        msgs = [f"**{row['user_name'] or row['agent_name']}**: {row['text']}" 
               for _, row in room_data.sort_values('createdAt').iterrows()]
        return '\n\n'.join(msgs)

# Create and display
explorer = RoomExplorer(df_memories)
display(pn.Column(explorer.param, explorer.messages))

In [ ]:
room_id = '5aa1d98d-5ae5-09c3-b060-c3359863ae0e'

In [ ]:
relationship_matrix

In [ ]:
df_schema[df_schema['table_name'].isin(['cache','logs','memories','rooms', 'knowledge', 'accounts'])]

Plot total memory count and total memory size per agent.

In [ ]:
import panel as pn
import param
import pandas as pd
from datetime import datetime

class RoomExplorer(param.Parameterized):
   room_id = param.ObjectSelector()
   
   def __init__(self, df_memories, **params):
       super().__init__(**params)
       self.df_memories = df_memories
       self.room_ids = df_memories['roomId'].unique()
       self.param.room_id.objects = self.room_ids
       
   def get_room_stats(self):
       room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
       
       stats = {
           'Total Memories': len(room_data),
           'Unique Participants': room_data['user_name'].nunique(),
           'Unique Agents': room_data['agent_name'].nunique(),
           'Date Range': f"{room_data['createdAt'].min()} to {room_data['createdAt'].max()}"
       }
       
       return pn.pane.HTML(f"""
       <div style="padding: 10px; background: #f5f5f5; border-radius: 5px;">
           <h3>Room Statistics</h3>
           {''.join(f'<p><b>{k}:</b> {v}</p>' for k,v in stats.items())}
       </div>
       """)
   
   def get_messages(self):
       room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
       room_data = room_data.sort_values('createdAt')
       
       messages_html = ''.join([
           f"""
           <div style="margin: 10px 0; padding: 10px; border: 1px solid #eee;">
               <p><b>{row['user_name'] or row['agent_name']}</b> at {row['createdAt']}</p>
               <p>{row['text']}</p>
           </div>
           """ for _, row in room_data.iterrows()
       ])
       
       return pn.pane.HTML(f"<div style='height: 500px; overflow-y: scroll;'>{messages_html}</div>")
   
   @param.depends('room_id')
   def view(self):
       if not self.room_id:
           return pn.pane.HTML("Select a room to begin")
           
       return pn.Column(
           self.get_room_stats(),
           self.get_messages()
       )

# Initialize and display
explorer = RoomExplorer(df_memories)
pn.Row(explorer.param, explorer.view).servable()

In [ ]:
agent_metrics = (
    df_memories
    .groupby('agent_name')
    .agg(
        count=('id', 'count'), 
        text_size=('text', lambda x: sum(len(y) for y in x if pd.notnull(y))),
    )
    .reset_index()
)

In [ ]:
agent_metrics

In [ ]:
import hvplot.pandas

agent_metrics = agent_metrics.sort_values('count', ascending=False)

# Create plots
plot1 = agent_metrics.hvplot.bar(
    x='agent_name', y='count', 
    title='Memory Count by Agent',
    color='royalblue',
    height=300, width=600
)

plot2 = agent_metrics.hvplot.bar(
    x='agent_name', y='text_size',
    title='Total Memory Size by Agent', 
    color='seagreen',
    height=300, width=600
)

plot1 + plot2

In [ ]:
agent_metrics

In [ ]:
# If everything works, let's set up SQLAlchemy
engine = create_engine(f'sqlite:///{db_path}')

# Test a simple query with the engine
df_knowledge = pd.read_sql_query("""
SELECT id, agentId, isMain, isShared, createdAt
FROM knowledge
""", engine)


# Get some basic stats
stats_query = """
SELECT 
    'memories' as table_name,
    COUNT(*) as count,
    COUNT(DISTINCT userId) as unique_users
FROM memories

UNION ALL

SELECT 
    'knowledge' as table_name,
    COUNT(*) as count,
    COUNT(DISTINCT agentId) as unique_users
FROM knowledge
"""

df_stats = pd.read_sql_query(stats_query, engine)

In [ ]:
df_knowledge